In [1]:
import fastplotlib as fpl
import os
import sys
import masknmf
import tifffile
import numpy as np
%load_ext autoreload

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),NVIDIA GeForce RTX 3090,DiscreteGPU,Vulkan,560.94
✅,NVIDIA GeForce RTX 3090,DiscreteGPU,D3D12,
✅,NVIDIA GeForce RTX 3090,DiscreteGPU,D3D12,
❗ limited,Microsoft Basic Render Driver,CPU,D3D12,
❌,NVIDIA GeForce RTX 3090/PCIe/SSE2,Unknown,OpenGL,4.6.0 NVIDIA 560.94


To silence this warning, use a fully namespaced name.


ModuleNotFoundError: No module named 'masknmf'

In [ ]:
data = tifffile.imread("../../demo_data/250206-UK6-1-F=4_power=5mW_reg_t_crop_s_full.tiff")

# If you don't have a good template estimate, run the generic template estimation procedure. End result is a PiecewiseRigidRegistrationStrategy object, used to register all frames to a template

In [ ]:
pwrigid_strategy = masknmf.PiecewiseRigidMotionCorrector(
    # num_blocks=[10, 10],
    # overlaps=[5, 5],
    # max_rigid_shifts=[15, 15],
    # max_deviation_rigid=[2, 2],
    # batch_size=500
    num_blocks=[25, 25],
    overlaps=[25, 25],
    max_rigid_shifts=[15, 15],
    max_deviation_rigid=[10, 10],
    batch_size=200
)

pwrigid_strategy.compute_template(data)

100%|██████████| 10/10 [00:03<00:00,  2.60it/s]


# Define a RegistrationArray that lazily loads motion corrected frames of the raw data

In [ ]:
moco_results = masknmf.RegistrationArray(data, pwrigid_strategy)

# Visualize the raw data, motion corrected data, template

In [ ]:
iw = fpl.ImageWidget(data = [data, moco_results],
                     names = ['raw', 'motion corrected'])
iw.show()

RFBOutputContext()

c:\Users\sterada\anaconda3\envs\masknmf\Lib\site-packages\fastplotlib\graphics\features\_base.py:18: UserWarning: casting float64 array to float32
  warn(f"casting {array.dtype} array to float32")


JupyterRenderCanvas(css_height='300.0px', css_width='500.0px')

## Access the shifts per block per frame

In [ ]:
shifts = moco_results.shifts[:]

# Visualize with fastplotlib imagewidget (will be updated with next fastplotlib update)

In [ ]:
# iw = fpl.ImageWidget(
#     data=[data, moco_results, pwrigid_strategy.template],
#     names = ['raw data', 'motion corrected', 'template'],
#     figure_shape=(1, 3),
#     cmap="viridis",
#     window_funcs={"t": (np.mean, 11)},
# )

# x, y = moco_results.block_centers.transpose(-1, 0, 1)
# u, v = moco_results.shifts[0].transpose(-1, 0, 1)

# # positions of each vector as [n_points, 2] array
# positions = np.column_stack([x.ravel(), y.ravel()])

# # directions of each vector as a [n_points, 2] array
# # scale down by 5 otherwise they're too big
# directions = np.column_stack([u.ravel(), v.ravel()]) / 5

# vector_field = iw.figure[0, 0].add_vector_field(
#     positions=positions,
#     directions=directions,
#     alpha=0.7,
#     alpha_mode="add",
#     color="w",
# )

# @iw.add_event_handler
# def update_vector_field(index):
#     t = index["t"]

#     u, v = moco_results.shifts[t].transpose(-1, 0, 1)
#     directions = np.column_stack([u.ravel(), v.ravel()])
    
#     vector_field.directions = directions / 5


# iw.show()